# Baseline of the baseline

In [16]:
# ──────────────────────────────────────────────────────────────
# 0.  Install minimal dependencies (uncomment if running fresh)
# %pip install scikit-learn==1.5.0 pandas numpy
# ──────────────────────────────────────────────────────────────

from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
import numpy as np, pandas as pd

## Load data ─ adapt the path if needed

In [17]:
df = pd.read_csv("train.csv")          # columns: id, excerpt, target, …

y       = df["target"].values
texts   = df["excerpt"].values

y[:5]
texts[:5]

array(['When the young people returned to the ballroom, it presented a decidedly changed appearance. Instead of an interior scene, it was a winter landscape.\r\nThe floor was covered with snow-white canvas, not laid on smoothly, but rumpled over bumps and hillocks, like a real snow field. The numerous palms and evergreens that had decorated the room, were powdered with flour and strewn with tufts of cotton, like snow. Also diamond dust had been lightly sprinkled on them, and glittering crystal icicles hung from the branches.\r\nAt each end of the room, on the wall, hung a beautiful bear-skin rug.\r\nThese rugs were for prizes, one for the girls and one for the boys. And this was the game.\r\nThe girls were gathered at one end of the room and the boys at the other, and one end was called the North Pole, and the other the South Pole. Each player was given a small flag which they were to plant on reaching the Pole.\r\nThis would have been an easy matter, but each traveller was obliged to 

## TF-IDF vectoriser: unigrams+bigrams, ignore rare words (<3 docs)

In [18]:
# TF-IDF vectoriser (term ferquency + term importance): unigrams+bigrams, ignore rare words (<3 docs)
tfidf = TfidfVectorizer(
        min_df=3,
        ngram_range=(1, 2),
        stop_words="english",# drop common fillers
        dtype=np.float32 # to save memory
)

## 5-fold cross-validation

In [19]:
# Ensure scikit-learn is installed and import KFold if needed

from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.empty(len(df))                # out-of-fold predictions holder

for fold, (train_idx, valid_idx) in enumerate(kf.split(texts, y)):
    X_train = tfidf.fit_transform(texts[train_idx])
    X_valid = tfidf.transform(texts[valid_idx])

    model = Ridge(alpha=1.0, random_state=42)   # α=1 is usually good
    model.fit(X_train, y[train_idx])

    oof[valid_idx] = model.predict(X_valid)
    rmse = root_mean_squared_error(y[valid_idx], oof[valid_idx])
    print(f"Fold {fold+1}: RMSE = {rmse:.4f}")

cv_rmse = root_mean_squared_error(y, oof)
print(f"\nCV mean RMSE: {cv_rmse:.4f}")

Fold 1: RMSE = 0.7413
Fold 2: RMSE = 0.7710
Fold 3: RMSE = 0.7664
Fold 4: RMSE = 0.7800
Fold 5: RMSE = 0.7422

CV mean RMSE: 0.7604


## Train on full data

In [20]:
# Train a final model on the full dataset
X_full   = tfidf.fit_transform(texts)
full_model = Ridge(alpha=1.0, random_state=42)
full_model.fit(X_full, y)

Ridge(random_state=42)

## Inference on the training set

In [21]:
# Inference on the training set
inference_vecs = tfidf.transform(df["excerpt"].values)
predicted_scores = full_model.predict(inference_vecs)

print("First 5 predicted scores:", predicted_scores[:5])

First 5 predicted scores: [-0.5363928  -0.46021146 -0.5465977  -1.1924889  -0.33898687]
